# Problema #12 — Estabilidad del modelo predictivo frente al Undersampling

## Versión operativa Jupyter — RUN ALL

## Ficha técnica de esta corrida

| Campo | Valor |
|---|---|
| Equipo | Victor Malsam / David Torres |
| Modalidad | Gerencial — Grupo A |
| Escenario | **Undersampling 5%** |
| Orden SEM1 | **4 de 5** |
| `training_pct` | **0.05** |
| Semilla | **198427** |
| Experimento | **P12_U005_S198427** |
| Entorno | **Jupyter / VM** |
| Modo | **RUN ALL** |    

### Variable experimental

La variable experimental del Problema #12 es:

`PARAM$trainingstrategy$training_pct`

Escenarios previstos: **100%, 40%, 10%, 5% y 1%**.

La **semilla** cambia como repetición experimental y debe mantenerse pareada entre escenarios.

### Configuración LightGBM que NO se modifica entre escenarios

Parámetros fijos:

- `max_bin = 31`
- `learning_rate = 0.03`
- `feature_fraction = 0.5`
- `num_iterations = 2048` como máximo de entrenamiento
- `early_stopping_rounds = 200`

Grid Search — mantener siempre la misma grilla:

- `num_leaves = 64, 128, 256, 512`
- `min_data_in_leaf = 64, 256, 512, 1024, 2048`
- Total: **20 combinaciones**

**Importante:** `num_leaves`, `min_data_in_leaf` y `niter/best_iter` ganadores son resultados de la optimización de cada corrida. No se fijan manualmente a partir de una corrida anterior.

### Cortes Kaggle

El workflow conserva los cortes originales:

`800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300`

Total: **11 cortes**.

### Adaptación Jupyter

Esta copia conserva la estructura del workflow del profesor, pero se quitaron los **dos bloques de código exclusivos de Google Colab**:

1. `from google.colab import drive` / `drive.mount(...)`
2. bloque `%%shell` que crea rutas `/content/...`, copia `kaggle.json` y descarga datasets.

En la VM/Jupyter se utilizan las rutas persistentes bajo `/home/ds/buckets/b1/...`.

> Regla de trabajo: no modificar el workflow experimental fuera de los cambios expresamente documentados para el Problema #12.


### Forma de ejecución de esta plantilla

1. Cambiar **solamente** `P12_SEMILLA` en la celda de configuración.
2. Guardar el notebook.
3. Ejecutar `Kernel -> Restart Kernel and Run All Cells`.
4. No ejecutar manualmente celdas intermedias.

El escenario de esta plantilla está bloqueado en **UNDERSAMPLING 5%**.
El identificador del experimento se genera automáticamente a partir del escenario y la semilla.


# 6 WorkFlow Gerencial, futuro=SEP

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
# RUN ALL - comprobacion de inicio del notebook
cat("Inicio de ejecucion del notebook P12 UND_005\n")


Inicio de ejecucion del notebook P12 UND_005


In [2]:
# limpio la memoria antes de comenzar una corrida completa
rm(list=ls(all.names=TRUE))
gc(full=TRUE, verbose=FALSE)

# tiempo inicial de la corrida completa
P12_INICIO <- Sys.time()
cat("Inicio:", format(P12_INICIO, "%a %b %d %X %Y"), "\n")


,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,668016,35.7,1479540,79.1,1479540,79.1
Vcells,1236216,9.5,8388608,64.0,1978697,15.1


Inicio: Wed Sep 09 18:01:24 2026 


In [3]:
# paquetes generales requeridos por el workflow
if( !require("data.table") ) install.packages("data.table")
require("data.table")

if( !require("R.utils") ) install.packages("R.utils")
require("R.utils")


Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach, load, save


R.utils v2.13.0 (2025-02-24 21:20:02 UTC) successfully loaded. See ?R.utils for help.


Attaching package: ‘R.utils’


The following object is masked from ‘package:utils’:

    timestamp


The following objects are masked from ‘package:base’:

    cat, commandArgs, getOption, isOpen, nullfile, parse, u

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [4]:
# ============================================================
# P12 - CONFIGURACION DE ESTA CORRIDA
# ============================================================
# CAMBIAR SOLAMENTE ESTA LINEA PARA UNA NUEVA REPETICION:
P12_SEMILLA <- 198427

# ------------------------------------------------------------
# CONFIGURACION BLOQUEADA DE ESTA PLANTILLA: UNDERSAMPLING 5%
# NO MODIFICAR PARA ESTA VERSION
# ------------------------------------------------------------
P12_ESCENARIO <- "UND_005"
P12_TRAINING_PCT <- 0.05
P12_CODIGO_ESCENARIO <- "U005"

# PARAM general del workflow
PARAM <- list()
PARAM$semilla_primigenia <- P12_SEMILLA
PARAM$experimento <- paste0("P12_", P12_CODIGO_ESCENARIO, "_S", P12_SEMILLA)
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"

# metadatos del experimento
PARAM$p12 <- list()
PARAM$p12$escenario <- P12_ESCENARIO
PARAM$p12$training_pct <- P12_TRAINING_PCT
PARAM$p12$version <- "RUNALL_V2"

PARAM$ejecucion <- list()
PARAM$ejecucion$inicio <- format(P12_INICIO, "%Y-%m-%d %H:%M:%S")

cat("Escenario :", P12_ESCENARIO, "\n")
cat("training_pct :", P12_TRAINING_PCT, "\n")
cat("Semilla :", P12_SEMILLA, "\n")
cat("Experimento :", PARAM$experimento, "\n")


Escenario : UND_005 
training_pct : 0.05 
Semilla : 198427 
Experimento : P12_U005_S198427 


#### Carpeta del Experimento

In [5]:
# ============================================================
# PREFLIGHT JUPYTER / VM - antes del trabajo pesado
# ============================================================
P12_BASE_EXP <- "/home/ds/buckets/b1/exp"
P12_BASE_DATASETS <- "/home/ds/buckets/b1/datasets"
P12_DATASET_PATH <- file.path(P12_BASE_DATASETS, PARAM$dataset)
P12_KAGGLE_JSON <- path.expand("~/.kaggle/kaggle.json")

if( !dir.exists(P12_BASE_EXP) ) stop("No existe: ", P12_BASE_EXP)
if( file.access(P12_BASE_EXP, 2) != 0 ) stop("Sin permiso de escritura en: ", P12_BASE_EXP)
if( !file.exists(P12_DATASET_PATH) ) stop("No existe el dataset: ", P12_DATASET_PATH)
if( Sys.which("kaggle") == "" ) stop("No se encontro el comando kaggle en la VM")
if( !file.exists(P12_KAGGLE_JSON) ) stop("No existe ~/.kaggle/kaggle.json")

# carpeta unica derivada automaticamente de escenario + semilla
experimento_folder <- paste0("WF", PARAM$experimento)
P12_EXPERIMENTO_DIR <- file.path(P12_BASE_EXP, experimento_folder)

# evita pisar o volver a enviar accidentalmente una corrida ya iniciada
if( dir.exists(P12_EXPERIMENTO_DIR) ) {
  stop("La carpeta de esta corrida ya existe: ", P12_EXPERIMENTO_DIR,
       "\nUse otra semilla o quite manualmente la corrida si realmente desea repetirla.")
}

dir.create(P12_EXPERIMENTO_DIR, recursive=FALSE, showWarnings=FALSE)
setwd(P12_EXPERIMENTO_DIR)

cat("PRECHECK OK\n")
cat("Directorio de corrida:", getwd(), "\n")


PRECHECK OK
Directorio de corrida: /home/ds/buckets/b1/exp/WFP12_U005_S198427 


In [6]:
# diagnostico de rutas de la VM/Jupyter
getwd()
Sys.info()[["user"]]

dir.exists("/home/ds/buckets/b1")
dir.exists("/home/ds/buckets/b1/exp")
dir.exists("/home/ds/buckets/b1/datasets")

file.access("/home/ds/buckets/b1/exp", 0)  # existe
file.access("/home/ds/buckets/b1/exp", 1)  # puede entrar
file.access("/home/ds/buckets/b1/exp", 2)  # puede escribir


[1] "/home/ds/buckets/b1/exp/WFP12_U005_S198427"

[1] "ds"

[1] TRUE

[1] TRUE

[1] TRUE

/home/ds/buckets/b1/exp 
                      0

/home/ds/buckets/b1/exp 
                      0

/home/ds/buckets/b1/exp 
                      0

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [7]:
# lectura del dataset
# dataset <- fread(paste0("/content/datasets/", PARAM$dataset))
dataset <- fread(
  paste0("/home/ds/buckets/b1/datasets/", PARAM$dataset)
)

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [8]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [9]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [10]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


In [11]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"           "foto_mes"                   
 [3] "internet"                    "cliente_edad"               
 [5] "cliente_antiguedad"          "mrentabilidad"              
 [7] "mrentabilidad_annual"        "mcomisiones"                
 [9] "mactivos_margen"             "mpasivos_margen"            
[11] "cproductos"                  "mcuenta_corriente"          
[13] "mcaja_ahorro"                "cdescubierto_preacordado"   
[15] "mcuentas_saldo"              "ctarjeta_visa_transacciones"
[17] "mtarjeta_visa_consumo"       "mtarjeta_master_consumo"    
[19] "mprestamos_personales"       "cpayroll_trx"               
[21] "mpayroll"                    "ccomisiones_mantenimiento"  
[23] "ccallcenter_transacciones"   "chomebanking_transacciones" 
[25] "ctrx_quarter"                "Master_status"              
[27] "Master_fechaalta"            "Master_mpagominimo"         
[29] "Visa_status"                 "Visa_fechaalta"             
[31] "Visa_mpagominimo"            "clase_ternaria"             
[33] "kmes"                        "mpayroll_sobre_edad"

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [12]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Fature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [13]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [14]:
ncol(dataset)
colnames(dataset)

[1] 158

[1] "numero_de_cliente"                  "foto_mes"                          
  [3] "internet"                           "cliente_edad"                      
  [5] "cliente_antiguedad"                 "mrentabilidad"                     
  [7] "mrentabilidad_annual"               "mcomisiones"                       
  [9] "mactivos_margen"                    "mpasivos_margen"                   
 [11] "cproductos"                         "mcuenta_corriente"                 
 [13] "mcaja_ahorro"                       "cdescubierto_preacordado"          
 [15] "mcuentas_saldo"                     "ctarjeta_visa_transacciones"       
 [17] "mtarjeta_visa_consumo"              "mtarjeta_master_consumo"           
 [19] "mprestamos_personales"              "cpayroll_trx"                      
 [21] "mpayroll"                           "ccomisiones_mantenimiento"         
 [23] "ccallcenter_transacciones"          "chomebanking_transacciones"        
 [25] "ctrx_quarter"                       "Master_status"                     
 [27] "Master_fechaalta"                   "Master_mpagominimo"                
 [29] "Visa_status"                        "Visa_fechaalta"                    
 [31] "Visa_mpagominimo"                   "clase_ternaria"                    
 [33] "kmes"                               "mpayroll_sobre_edad"               
 [35] "internet_lag1"                      "cliente_edad_lag1"                 
 [37] "cliente_antiguedad_lag1"            "mrentabilidad_lag1"                
 [39] "mrentabilidad_annual_lag1"          "mcomisiones_lag1"                  
 [41] "mactivos_margen_lag1"               "mpasivos_margen_lag1"              
 [43] "cproductos_lag1"                    "mcuenta_corriente_lag1"            
 [45] "mcaja_ahorro_lag1"                  "cdescubierto_preacordado_lag1"     
 [47] "mcuentas_saldo_lag1"                "ctarjeta_visa_transacciones_lag1"  
 [49] "mtarjeta_visa_consumo_lag1"         "mtarjeta_master_consumo_lag1"      
 [51] "mprestamos_personales_lag1"         "cpayroll_trx_lag1"                 
 [53] "mpayroll_lag1"                      "ccomisiones_mantenimiento_lag1"    
 [55] "ccallcenter_transacciones_lag1"     "chomebanking_transacciones_lag1"   
 [57] "ctrx_quarter_lag1"                  "Master_status_lag1"                
 [59] "Master_fechaalta_lag1"              "Master_mpagominimo_lag1"           
 [61] "Visa_status_lag1"                   "Visa_fechaalta_lag1"               
 [63] "Visa_mpagominimo_lag1"              "kmes_lag1"                         
 [65] "mpayroll_sobre_edad_lag1"           "internet_lag2"                     
 [67] "cliente_edad_lag2"                  "cliente_antiguedad_lag2"           
 [69] "mrentabilidad_lag2"                 "mrentabilidad_annual_lag2"         
 [71] "mcomisiones_lag2"                   "mactivos_margen_lag2"              
 [73] "mpasivos_margen_lag2"               "cproductos_lag2"                   
 [75] "mcuenta_corriente_lag2"             "mcaja_ahorro_lag2"                 
 [77] "cdescubierto_preacordado_lag2"      "mcuentas_saldo_lag2"               
 [79] "ctarjeta_visa_transacciones_lag2"   "mtarjeta_visa_consumo_lag2"        
 [81] "mtarjeta_master_consumo_lag2"       "mprestamos_personales_lag2"        
 [83] "cpayroll_trx_lag2"                  "mpayroll_lag2"                     
 [85] "ccomisiones_mantenimiento_lag2"     "ccallcenter_transacciones_lag2"    
 [87] "chomebanking_transacciones_lag2"    "ctrx_quarter_lag2"                 
 [89] "Master_status_lag2"                 "Master_fechaalta_lag2"             
 [91] "Master_mpagominimo_lag2"            "Visa_status_lag2"                  
 [93] "Visa_fechaalta_lag2"                "Visa_mpagominimo_lag2"             
 [95] "kmes_lag2"                          "mpayroll_sobre_edad_lag2"          
 [97] "internet_delta1"                    "internet_delta2"                   
 [99] "cliente_edad_delta1"                "cliente_edad_delta2"               
[1

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [15]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset reducido de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202109  obviamente completo

* final_train =  [ 202005, 202107 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202107   completo, sin undersampling
   * training = [ 202005, 202106 ]  donde se consideran el 100% de los CONTINUA

In [16]:
PARAM$trainingstrategy$validate <- c(202107)

PARAM$trainingstrategy$training <- c(
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- P12_TRAINING_PCT


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [17]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [18]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

Esta celda tarda en correr interminables 7 minutos en Colab
<br> ya que debe instalar la librería de LightGBM

In [19]:
# preparo para que se puede hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]

# ============================================================
# P12 - REGISTRO DEL EFECTO DEL UNDERSAMPLING
# ============================================================

PARAM$p12$registros <- list()

# cantidad de registros disponibles ANTES del undersampling
PARAM$p12$registros$training_antes <- dataset[
  foto_mes %in% PARAM$trainingstrategy$training,
  .N
]

# positivos antes del undersampling
PARAM$p12$registros$positivos_antes <- dataset[
  foto_mes %in% PARAM$trainingstrategy$training &
  clase01 == 1,
  .N
]

# negativos / CONTINUA antes del undersampling
PARAM$p12$registros$negativos_antes <- dataset[
  foto_mes %in% PARAM$trainingstrategy$training &
  clase01 == 0,
  .N
]

# cantidad de registros que realmente entran al training
PARAM$p12$registros$training_despues <- dataset[
  fold_train == TRUE,
  .N
]

# positivos que quedaron en training
PARAM$p12$registros$positivos_despues <- dataset[
  fold_train == TRUE & clase01 == 1,
  .N
]

# negativos que quedaron en training
PARAM$p12$registros$negativos_despues <- dataset[
  fold_train == TRUE & clase01 == 0,
  .N
]

# porcentaje real de registros conservados
PARAM$p12$registros$porcentaje_real_conservado <- round(
  100 *
  PARAM$p12$registros$training_despues /
  PARAM$p12$registros$training_antes,
  4
)

# muestro el control durante la corrida
cat("\nP12 - REGISTROS DE TRAINING\n")
cat("Antes           :", PARAM$p12$registros$training_antes, "\n")
cat("Después         :", PARAM$p12$registros$training_despues, "\n")
cat("Positivos antes :", PARAM$p12$registros$positivos_antes, "\n")
cat("Positivos después:", PARAM$p12$registros$positivos_despues, "\n")
cat("Negativos antes :", PARAM$p12$registros$negativos_antes, "\n")
cat("Negativos después:", PARAM$p12$registros$negativos_despues, "\n")
cat("% real conservado:", PARAM$p12$registros$porcentaje_real_conservado, "%\n")

  
if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)


P12 - REGISTROS DE TRAINING
Antes           : 179449 
Después         : 10524 
Positivos antes : 1673 
Positivos después: 1673 
Negativos antes : 177776 
Negativos después: 8851 
% real conservado: 5.8646 %


Loading required package: lightgbm



In [20]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13202

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [21]:
PARAM$trainingstrategy$training_pct

[1] 0.05

In [22]:
# checkpoint en disco dentro de la carpeta de ESTA corrida
saveRDS(
  dataset,
  file = "dataset_preprocesado_gerencial.rds"
)


In [23]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [24]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [25]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 50 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [26]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Wed Sep 09 18:01:47 2026  64, 64 niter 409 AUC 0.948410813624386

Wed Sep 09 18:01:56 2026  64, 256 niter 1815 AUC 0.949474391923398

Wed Sep 09 18:01:58 2026  64, 512 niter 596 AUC 0.942577615840897

Wed Sep 09 18:02:03 2026  64, 1024 niter 2039 AUC 0.939782469609725

Wed Sep 09 18:02:07 2026  64, 2048 niter 2021 AUC 0.932050706485789

Wed Sep 09 18:02:14 2026  128, 64 niter 465 AUC 0.948690588503204

Wed Sep 09 18:02:22 2026  128, 256 niter 1815 AUC 0.949474391923398

Wed Sep 09 18:02:24 2026  128, 512 niter 596 AUC 0.942577615840897

Wed Sep 09 18:02:29 2026  128, 1024 niter 2039 AUC 0.939782469609725

Wed Sep 09 18:02:34 2026  128, 2048 niter 2021 AUC 0.932050706485789

Wed Sep 09 18:02:41 2026  256, 64 niter 513 AUC 0.948213886810458

Wed Sep 09 18:02:49 2026  256, 256 niter 1815 AUC 0.949474391923398

Wed Sep 09 18:02:53 2026  256, 512 niter 596 AUC 0.942577615840897

Wed Sep 09 18:03:01 2026  256, 1024 niter 2039 AUC 0.939782469609725

Wed Sep 09 18:03:06 2026  256, 2048 niter 2

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [27]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9484108,409
64,256,0.9494744,1815
64,512,0.9425776,596
64,1024,0.9397825,2039
64,2048,0.9320507,2021
128,64,0.9486906,465
128,256,0.9494744,1815
128,512,0.9425776,596
128,1024,0.9397825,2039


In [28]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 64

$min_data_in_leaf
[1] 256

$num_iterations
[1] 1815

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en optimización de hiperparámetros

In [29]:
PARAM$trainingstrategy$final_train <- c( 202107,
  202106, 202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 192651

##### Final Training Hyperparameters

In [30]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

In [31]:
final_model <- lgb.train(
  data= dfinal_train,
  param= param_final,
  verbose= -100
)

In [32]:
# grabo a disco el modelo en un formato para seres humanos ... ponele ...

lgb.save(final_model, "modelo.txt")

In [33]:
# ahora imprimo la importancia de variables

tb_importancia <- as.data.table(lgb.importance(final_model))
archivo_importancia <- "impo.txt"

fwrite( tb_importancia,
  file= archivo_importancia,
  sep= "\t"
)

#### Scoring

Aplico el modelo final a los datos del futuro

In [34]:
PARAM$trainingstrategy$future <- c(202109)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]

In [35]:
# aplico final_model   a dfuture

prediccion <- predict(
  final_model,
  data.matrix(dfuture[, campos_buenos, with= FALSE])
)

##### Tabla Prediccion

In [36]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := prediccion]

# grabo las probabilidad del modelo
#  me va a ser util para hacer Ensembles de modelos
fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)

#### Kaggle Competition Submit

Genero las salidas y hago los submits a Kaggle
<br>El notebook esta preparado para la Modalidad Gerencial, los analistas deben hacer cambios.
<br> Los analistas deben cambiar **competencia** a SU competencia  "data-mining-analista-jr-2025-a"   o  la original "data-mining-analista-sr-2025-a"  para los Senior
<br> Los cortes  dependen de la cantidad de registros, multiplicar por 2 para los Analistas Jr y por 10 para los Analista Sr

Los Analista Sr luego de meditar cuidadosamente reducirán la cantidad de cortes

In [37]:
# genero archivos con los  "envios" mejores
# suba TODOS los archivos a Kaggle

# PARAM$kaggle$competencia <- "data-mining-manager-2026-b"
PARAM$kaggle$competencia <- "utn-2026-virtual-mgr"
PARAM$kaggle$cortes <- seq(800, 1300, by = 50)

# ordeno por probabilidad descendente
setorder(tb_prediccion, -prob)

dir.create("kaggle")

for (envios in PARAM$kaggle$cortes) {

  tb_prediccion[, Predicted := 0L] # seteo inicial a 0
  tb_prediccion[1:envios, Predicted := 1L] # marclo los primeros

  archivo_kaggle <- paste0("./kaggle/KA", PARAM$experimento, "_", envios, ".csv")

  # grabo el archivo
  fwrite(tb_prediccion[, list(numero_de_cliente, Predicted)],
    file= archivo_kaggle,
    sep= ","
  )

  # subida a Kaggle, armo la linea de comando
  comando <- "kaggle competitions submit"
  competencia <- paste("-c", PARAM$kaggle$competencia)
  arch <- paste( "-f", archivo_kaggle)

  mensaje <- paste0("-m 'envios=", envios,
  "  semilla=", PARAM$semilla_primigenia,
    "'" )

  linea <- paste( comando, competencia, arch, mensaje)

  Sys.sleep(30)
  salida <- system(linea, intern=TRUE) # el submit a Kaggle
  cat(salida, "\n")
}

55 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
54 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
53 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
52 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
51 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
50 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
49 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
48 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
47 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
46 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 
45 submissions remaining today. Successfully submitted to UTN 2026 virtual mgr 


In [38]:
# ============================================================
# CIERRE - tiempos y PARAM.yml
# ============================================================
P12_FIN <- Sys.time()
PARAM$ejecucion$fin <- format(P12_FIN, "%Y-%m-%d %H:%M:%S")
PARAM$ejecucion$duracion_segundos <- as.numeric(difftime(P12_FIN, P12_INICIO, units="secs"))
PARAM$ejecucion$duracion_minutos <- round(PARAM$ejecucion$duracion_segundos / 60, 2)
PARAM$ejecucion$directorio <- getwd()

if( !require("yaml") ) install.packages("yaml")
require("yaml")

write_yaml(PARAM, file="PARAM.yml")
cat("PARAM.yml guardado correctamente\n")


Loading required package: yaml



PARAM.yml guardado correctamente


In [39]:
# ============================================================
# RESUMEN FINAL DE LA CORRIDA
# ============================================================
cat("\n============================================================\n")
cat("P12 - CORRIDA FINALIZADA\n")
cat("============================================================\n")
cat("Escenario       :", PARAM$p12$escenario, "\n")
cat("training_pct    :", PARAM$trainingstrategy$training_pct, "\n")
cat("Semilla         :", PARAM$semilla_primigenia, "\n")
cat("Experimento     :", PARAM$experimento, "\n")
cat("AUC Grid Search :", PARAM$out$lgbm$AUC, "\n")
cat("Ganadores       :\n")
print(PARAM$out$lgbm$mejores_hiperparametros)
cat("Inicio          :", PARAM$ejecucion$inicio, "\n")
cat("Fin             :", PARAM$ejecucion$fin, "\n")
cat("Duracion minutos:", PARAM$ejecucion$duracion_minutos, "\n")
cat("Training antes  :", PARAM$p12$registros$training_antes, "\n")
cat("Training despues:", PARAM$p12$registros$training_despues, "\n")
cat("% real conservado:", PARAM$p12$registros$porcentaje_real_conservado, "%\n")
cat("Cortes Kaggle   :", paste(PARAM$kaggle$cortes, collapse=", "), "\n")
cat("============================================================\n")



P12 - CORRIDA FINALIZADA
Escenario       : UND_005 
training_pct    : 0.05 
Semilla         : 198427 
Experimento     : P12_U005_S198427 
AUC Grid Search : 0.9494744 
Ganadores       :
$num_leaves
[1] 64

$min_data_in_leaf
[1] 256

$num_iterations
[1] 1815

Inicio          : 2026-09-09 18:01:24 
Fin             : 2026-09-09 18:10:22 
Duracion minutos: 8.96 
Training antes  : 179449 
Training despues: 10524 
% real conservado: 5.8646 %
Cortes Kaggle   : 800, 850, 900, 950, 1000, 1050, 1100, 1150, 1200, 1250, 1300 
